# QuickQuake catalog visualization

Figures to look at the results of a QuickQuake run. All the logic lives in `qq_helpers.py`;
this notebook only sets paths, loads the results, calls plotting functions and (optionally)
saves the figures in `DATA_ROOT/images/visualization/`. Edit the **Settings** cell, then run all
cells. The analysis figures (magnitude of completeness, FI, comparison with reference catalogs,
DBSCAN clusters) are in `catalog_analysis.ipynb`.

**Kernel:** the QuickQuake conda environment (the one `install.sh` creates). In VS Code select it as the kernel; in a terminal run `conda activate <environment>` and then `jupyter lab`.

| Section | Figures |
|---|---|
| 2 | Event counts, 2-D map, stations, zoomed map, timeline with spans, cross section |
| 3 | 3-D views (time colours, monthly, interactive) |
| 4 | Waveforms: move-outs and spectrograms |
| 5 | Location changes GaMMA -> HypoInverse -> HypoDD |

In [ ]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio

# This notebook lives in quickquake/, next to qq_helpers.py
HELPERS_DIR = Path.cwd().resolve()
if str(HELPERS_DIR) not in sys.path:
    sys.path.insert(0, str(HELPERS_DIR))

import qq_helpers as h
importlib.reload(h)
print("helpers:", h.__file__)

pio.renderers.default = "plotly_mimetype+notebook_connected"   # interactive figures in VS Code and Jupyter

## 0. Settings
Everything you normally edit is in this cell.

In [ ]:
# ---------------- results of a QuickQuake run ----------------
# Relative paths are relative to this notebook's folder (quickquake/).
DATA_ROOT = Path("../examples/pavlof_2021/data").resolve()   # the "data" folder next to your QQ_driver.py
NAMEBASE = "GAMMA"                                           # same as location_namebase in QQ_driver.py
REGION = {"name": "Pavlof", "lat": 55.4133, "lon": -161.8903}   # volcano / region centre (degrees)
RADIUS_KM = None              # keep only events within this distance (km) of the centre; None = keep all
START = "2021-07-28T00:00:00Z"
END   = "2021-07-30T00:00:00Z"

# ---------------- what to draw ----------------
SAVE_FIGURES = True           # save into DATA_ROOT/images/visualization/
DEPTH_MAX_KM = 40.0           # deepest event shown in depth-coloured / 3-D figures (km)
N_WAVEFORM_EVENTS = 3         # events drawn in the move-out and spectrogram sections
SPANS = [                     # (start, end, label) periods for the timelines; [] = none
    ("2021-08-05", "2021-08-06", "Ash emissions"),
    ("2021-11-08", "2021-12-31", "Intermittent lahars"),
]
PROFILE = None                # cross section: dict(start_lon, start_lat, end_lon, end_lat, label_start, label_end, swath_km)

# ---------------- example: Aniakchak ----------------
# DATA_ROOT = Path("~/work/research/quickquake_work/paper2/aniakchak_0.1").expanduser()
# REGION = {"name": "Aniakchak", "lat": 56.9058, "lon": -158.209}
# RADIUS_KM = 20.0
# START, END = "2021-01-01T00:00:00Z", "2024-12-31T23:59:59Z"
# DEPTH_MAX_KM = 13.0
# SPANS = [("2021-08-01", "2021-08-02", "Ash resuspension"), ("2023-03-01", "2023-08-01", "Uplift-AVO"),
#          ("2022-10-01", "2023-06-30", "Uplift (Delgado, 2026)")]
# PROFILE = dict(start_lon=-158.2729, start_lat=56.87708, end_lon=-157.9812, end_lat=56.87708,
#                label_start="C'", label_end="C", swath_km=5.0)

# ---------------- paths inside DATA_ROOT (standard QuickQuake layout) ----------------
MERGED = DATA_ROOT / "merged"
CATALOG_CSV   = MERGED / f"output/{NAMEBASE}_hypodd_catalog_qcfiltered.csv"   # final QuickQuake catalog
HYPODD_CSV    = MERGED / f"output/{NAMEBASE}_hypodd_catalog.csv"
HYPOINV_CSV   = MERGED / f"output/{NAMEBASE}_hypoinv_good.csv"
GAMMA_CSV     = MERGED / f"input/{NAMEBASE}_catalog_cleaned.csv"
PICKS_CSV     = MERGED / f"input/{NAMEBASE}_picks_cleaned.csv"
STATIONS_JSON = MERGED / f"input/{NAMEBASE}_station_list.json"
WAVEFORM_INDEX = DATA_ROOT / "waveform_index.pkl"


IMAGE_DIR = h.ensure_dir(DATA_ROOT / "images" / "visualization")


def save(fig, name, **kwargs):
    """Save a figure into IMAGE_DIR when SAVE_FIGURES is True."""
    if SAVE_FIGURES and fig is not None:
        out = IMAGE_DIR / name
        if hasattr(fig, "write_html"):
            fig.write_html(out, include_plotlyjs="cdn")
        else:
            fig.savefig(out, bbox_inches="tight", **kwargs)
        print("saved:", out)

## 1. Load the catalog and the stations

In [ ]:
catc = h.CatalogCols(time="time", lat="latitude", lon="longitude", depth_km="depth_km", mag="magnitude")
stc = h.StationCols(station_id="station", lat="latitude", lon="longitude", elev_km="elev_km")

df_catalog = h.load_catalog_csv(CATALOG_CSV, catc=catc)
df_stations = h.load_stations_json(STATIONS_JSON)
print(f"catalog: {len(df_catalog)} events | stations: {len(df_stations)}")

if RADIUS_KM is not None:
    df_catalog = h.filter_catalog_by_radius(df_catalog, center_lat=REGION["lat"], center_lon=REGION["lon"],
                                            radius_km=RADIUS_KM, lat_col=catc.lat, lon_col=catc.lon)
    print(f"within {RADIUS_KM:g} km of {REGION['name']}: {len(df_catalog)} events "
          f"(farthest {df_catalog['r_km'].max():.2f} km)")

print("time range:", df_catalog[catc.time].min(), "->", df_catalog[catc.time].max())

## 2. Catalog maps and timelines

In [ ]:
fig, ax, counts = h.plot_event_count_time(catalog=df_catalog, time_col=catc.time, start=START, end=END,
                                          freq="D", figsize=(12, 5))
save(fig, "event_count_daily.svg")
plt.show()

In [ ]:
fig, ax = h.plot_mapview_seismicity(catalog=df_catalog, stations=df_stations, start=START, end=END,
                                    depth_vmin=0.0, depth_vmax=DEPTH_MAX_KM, region_name=REGION["name"],
                                    relief_zoom=12, figsize=(15, 8), savefig=False)
save(fig, "mapview_seismicity.svg")
plt.show()

In [ ]:
fig, ax = h.plot_mapview_basemap_stations(catalog=df_catalog, stations=df_stations, start=START, end=END,
                                          relief_zoom=12, figsize=(15, 8), savefig=False)
save(fig, "mapview_stations_only.svg")
plt.show()

In [ ]:
fig, ax, meta = h.plot_mapview_seismicity_zoom(catalog=df_catalog, start=START, end=END,
                                               depth_vmin=0.0, depth_vmax=DEPTH_MAX_KM, cmap_name="inferno_r",
                                               relief_zoom=12, step_lon=0.2, step_lat=0.1, figsize=(15, 8),
                                               zoom_factor=3, show=False)
save(fig, "mapview_zoom.svg")
plt.show()
print(meta)

In [ ]:
fig, ax, meta = h.plot_timeline_histogram(catalog=df_catalog, start=START, end=END, spans=SPANS,
                                          time_col=catc.time, fi_col="FI", n_bins=90, figsize=(14, 4),
                                          show_legend=bool(SPANS), savefig=False)
save(fig, "timeline_histogram.svg")
plt.show()

In [ ]:
if PROFILE is None:
    print("cross section skipped: set PROFILE in the Settings cell")
else:
    fig, ax = h.plot_mapview_seismicity(catalog=df_catalog, stations=df_stations, start=START, end=END,
                                        depth_vmin=0.0, depth_vmax=DEPTH_MAX_KM, region_name=REGION["name"],
                                        relief_zoom=12, figsize=(15, 8), savefig=False)
    h.add_profile_line(ax=ax, start_lon=PROFILE["start_lon"], start_lat=PROFILE["start_lat"],
                       end_lon=PROFILE["end_lon"], end_lat=PROFILE["end_lat"],
                       start_label=PROFILE["label_start"], end_label=PROFILE["label_end"],
                       line_color="red", line_width=3.0, marker_size=80, fontsize=16)
    save(fig, "mapview_seismicity_with_crosssection.svg")
    plt.show()

    fig, ax, meta = h.plot_cross_section_seismicity(
        catalog=df_catalog, start=START, end=END, catc=catc,
        start_lon=PROFILE["start_lon"], start_lat=PROFILE["start_lat"],
        end_lon=PROFILE["end_lon"], end_lat=PROFILE["end_lat"], swath_km=PROFILE["swath_km"],
        depth_vmin=0.0, depth_vmax=DEPTH_MAX_KM, depth_max_plot=DEPTH_MAX_KM, cmap_name="inferno_r",
        x_label_start=PROFILE["label_start"], x_label_end=PROFILE["label_end"], figsize=(12, 6))
    save(fig, "cross_section.svg")
    plt.show()
    print(meta)

## 3. 3-D views

In [ ]:
fig, ax, meta = h.plot_mapview3d_timecolor(catalog=df_catalog, start=START, end=END, depth_max=DEPTH_MAX_KM,
                                           dem_res="15s", exagg=1.0, view_elev=10.0, view_azim=265.0,
                                           figsize=(12, 9), savefig=False, catc=catc)
save(fig, "mapview3d_timecolor.svg")
plt.show()

In [ ]:
fig, ax, meta = h.plot_mapview3d_months(catalog=df_catalog, start=START, end=END, depth_max=DEPTH_MAX_KM,
                                        exagg=4, view_elev=4, view_azim=34, dem_res="15s", savefig=False)
save(fig, "mapview3d_months.svg")
plt.show()
print("events plotted:", meta["n_events"])

In [ ]:
fig = h.plot_catalog_3d_topo_seismicity(catalog=df_catalog, catc=catc, mode="time", dem_res="15s",
                                        region_name=REGION["name"], show=False)
save(fig, "catalog_3d_interactive.html")
fig.show()

## 4. Waveforms: move-outs and spectrograms
Needs the waveform chunks of the run (`DATA_ROOT/<chunk>/waveforms/*.mseed`) and the cleaned picks.

In [ ]:
has_waveforms = PICKS_CSV.exists() and (WAVEFORM_INDEX.exists() or any(DATA_ROOT.glob("*/waveforms/*.mseed")))
if not has_waveforms:
    print("waveform sections skipped: no picks file or no waveforms under DATA_ROOT")
else:
    wf_catalog, wf_picks, wf_stations = h.load_inputs(CATALOG_CSV, PICKS_CSV, STATIONS_JSON)
    if WAVEFORM_INDEX.exists():
        wf_index = h.WaveformIndex.load(WAVEFORM_INDEX)
    else:
        wf_index = h.WaveformIndex.build([DATA_ROOT], glob_pattern="**/waveforms/*.mseed")
        wf_index.save(WAVEFORM_INDEX)
    stream_cache = h.StreamCache()
    print("indexed mseed files:", len(wf_index.df))

In [ ]:
if has_waveforms:
    for row_idx, row_event in wf_catalog.iloc[:N_WAVEFORM_EVENTS].iterrows():
        fig, ax, meta = h.plot_event_moveout_pretty(
            row_event=row_event, picks=wf_picks, stations=wf_stations, cache=stream_cache, wf_index=wf_index,
            pre_s=0, post_s=40, freqmin=1.0, freqmax=5.0, figsize=(10, 7))
        save(fig, f"moveout_event_{row_idx}.svg")
        plt.show()
        print(row_idx, "stations used:", len(meta["stations_used"]))

In [ ]:
if has_waveforms:
    for row_idx, row_event in wf_catalog.iloc[:N_WAVEFORM_EVENTS].iterrows():
        fig, axes, meta = h.plot_event_spectrograms_vertical_picks_only(
            row_event=row_event, picks=wf_picks, stations=wf_stations, cache=stream_cache, wf_index=wf_index,
            pre_s=30, post_s=90, window_s=2, overlap_frac=0.8, freq_min=0, freq_max=20)
        save(fig, f"spectrograms_event_{row_idx}.png", dpi=200)
        plt.show()

## 5. Location changes between pipeline stages
The same events located by GaMMA, HypoInverse and HypoDD; each segment joins two locations of one event.

In [ ]:
# HypoDD stage = the final QC-filtered catalog (as in the paper figures); use HYPODD_CSV for all relocated events
df_loc = h.build_location_evolution_df(GAMMA_CSV, HYPOINV_CSV, CATALOG_CSV, start=START, end=END)
print(f"{len(df_loc)} events in all three stages | median horizontal shift: "
      f"GaMMA->HypoInverse {df_loc['shift_gamma_hypoinv_km'].median():.2f} km, "
      f"HypoInverse->HypoDD {df_loc['shift_hypoinv_hypodd_km'].median():.2f} km")
fig, axes, drawn = h.plot_location_evolution_map(df_loc, max_events=1200, pad_lon=0.02, pad_lat=0.02,
                                                 x_tick_label_size=15.3, y_tick_label_size=18, legend_fontsize=13)
save(fig, "location_changes_gamma_hypoinv_hypodd.pdf")
plt.show()